# Mapa completo: tudo o que dá para obter

**O que este notebook faz:** percorre **todas as bases** que a biblioteca PySUS
oferece e mostra, com dados atuais do catálogo, o que existe em cada uma —
período coberto, estados e grupos de informação.

Use-o como referência: quando quiser saber "existe dado de X?", rode este
notebook em vez de procurar na documentação.

**Tempo estimado:** 4 a 6 minutos (só consulta catálogos, não baixa dados).

## Preparação

In [2]:
%pip install pysus==2.10.6 nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()
print("Ambiente pronto.")

Note: you may need to restart the kernel to use updated packages.
Ambiente pronto.


## As nove bases da biblioteca

| Função | Base | Recorte exigido |
|---|---|---|
| `sinan()` | Doenças de notificação | agravo + ano (nacional) |
| `sim()` | Mortalidade | estado + ano |
| `sinasc()` | Nascidos vivos | estado + ano |
| `sih()` | Internações hospitalares | estado + ano + mês |
| `sia()` | Produção ambulatorial | estado + ano + mês |
| `cnes()` | Estabelecimentos, leitos, profissionais | estado + ano + mês + **grupo** |
| `pni()` | Imunizações | estado + ano |
| `ciha()` | Atendimentos além do SUS | estado + ano + mês + `group=None` |
| `ibge()` | População e indicadores sociais | ano |

## Levantamento automático

A célula abaixo consulta o catálogo de cada base e resume o que há.

In [3]:
from pysus import list_files
import pandas as pd

BASES = ["sinan", "sim", "sinasc", "sih", "sia", "cnes", "pni", "ciha", "ibge"]

resumo = []
for base in BASES:
    catalogo = list_files(dataset=base)
    nomes = [n.split("\\")[-1] for n in catalogo["name"]]
    anos = sorted({int(a) for a in catalogo["year"].dropna()}) if "year" in catalogo else []
    ufs = {u for u in catalogo["state"].dropna()} if "state" in catalogo else set()
    largura = 4 if base in ("sinan", "ibge") else 2
    grupos = sorted({n[:largura] for n in nomes})

    resumo.append({
        "Base": base.upper(),
        "Arquivos": len(catalogo),
        "De": anos[0] if anos else "-",
        "Até": anos[-1] if anos else "-",
        "UFs": len(ufs),
        "Grupos": len(grupos),
    })

pd.DataFrame(resumo)

,Base,Arquivos,De,Até,UFs,Grupos
0,SINAN,1106,1999,2026,1,61
1,SIM,837,1979,2026,29,2
2,SINASC,755,1994,2026,29,2
3,SIH,11050,1992,2026,28,7
4,SIA,50069,1994,2026,30,13
5,CNES,80192,2005,2026,27,14
6,PNI,1523,1994,2026,30,3
7,CIHA,4777,2011,2026,26,1
8,IBGE,152,1980,2070,0,8


## Os grupos de cada base

Cada base divide seus dados em **grupos** — conjuntos com colunas diferentes.
Saber qual grupo você precisa evita baixar centenas de megabytes à toa.

In [4]:
GRUPOS_CONHECIDOS = {
    "cnes": {
        "LT": "Leitos", "ST": "Estabelecimentos", "PF": "Profissionais",
        "EQ": "Equipamentos", "SR": "Serviços especializados",
        "HB": "Habilitações", "EP": "Equipes", "IN": "Incentivos",
        "RC": "Regras contratuais", "DC": "Dados complementares",
        "GM": "Gestão e metas", "EF": "Estabelecimentos filantrópicos",
        "EE": "Estabelecimento de ensino",
    },
    "sih": {
        "RD": "AIH reduzida (uma linha por internação)",
        "SP": "Serviços profissionais", "RJ": "AIH rejeitada",
        "ER": "AIH com erro", "CH": "Cadastro hospitalar",
        "CM": "Comunicação de movimentação",
    },
    "sia": {
        "PA": "Produção ambulatorial (principal)",
        "BI": "Boletim de produção individualizada",
        "AB": "APAC de cirurgia bariátrica", "AM": "APAC de medicamentos",
        "AN": "APAC de nefrologia", "AQ": "APAC de quimioterapia",
        "AR": "APAC de radioterapia", "AD": "APAC de laudos diversos",
        "AT": "APAC de tratamento", "AC": "APAC de confecção de fístula",
        "PS": "RAAS psicossocial", "SA": "Serviços especializados",
    },
    "sim": {"DO": "Declarações de óbito"},
    "sinasc": {"DN": "Declarações de nascido vivo", "DNR": "Retroativos"},
    "pni": {"CP": "Cobertura por município", "DP": "Doses aplicadas"},
    "ciha": {"CIHA": "Comunicação de internação hospitalar e ambulatorial"},
    "ibge": {
        "POPT": "População total", "POPB": "População por faixas",
        "PROJ": "Projeções populacionais", "ESCA": "Escolaridade",
        "ESCB": "Escolaridade (detalhe)", "ALFB": "Alfabetização",
        "REND": "Renda", "IDOS": "População idosa",
    },
}

for base, grupos in GRUPOS_CONHECIDOS.items():
    print(f"\n{base.upper()}")
    for codigo, descricao in grupos.items():
        print(f"   {codigo:<5} {descricao}")


CNES
   LT    Leitos
   ST    Estabelecimentos
   PF    Profissionais
   EQ    Equipamentos
   SR    Serviços especializados
   HB    Habilitações
   EP    Equipes
   IN    Incentivos
   RC    Regras contratuais
   DC    Dados complementares
   GM    Gestão e metas
   EF    Estabelecimentos filantrópicos
   EE    Estabelecimento de ensino

SIH
   RD    AIH reduzida (uma linha por internação)
   SP    Serviços profissionais
   RJ    AIH rejeitada
   ER    AIH com erro
   CH    Cadastro hospitalar
   CM    Comunicação de movimentação

SIA
   PA    Produção ambulatorial (principal)
   BI    Boletim de produção individualizada
   AB    APAC de cirurgia bariátrica
   AM    APAC de medicamentos
   AN    APAC de nefrologia
   AQ    APAC de quimioterapia
   AR    APAC de radioterapia
   AD    APAC de laudos diversos
   AT    APAC de tratamento
   AC    APAC de confecção de fístula
   PS    RAAS psicossocial
   SA    Serviços especializados

SIM
   DO    Declarações de óbito

SINASC
   DN    D

## Os agravos do SINAN

O SINAN é a única base organizada por doença. Estes são os agravos com dados
no ano mais recente:

In [5]:
ANO = 2024
catalogo = list_files(dataset="sinan", year=ANO)
agravos = sorted({n.split("\\")[-1][:4] for n in catalogo["name"]})

NOMES = {
    "DENG": "Dengue", "CHIK": "Chikungunya", "ZIKA": "Zika",
    "TUBE": "Tuberculose", "HANS": "Hanseníase", "LEPT": "Leptospirose",
    "MALA": "Malária", "MENI": "Meningite", "VIOL": "Violência",
    "ANIM": "Acidente por animal peçonhento", "ACGR": "Acidente de trabalho grave",
    "ACBI": "Acidente biológico", "ESQU": "Esquistossomose",
    "CHAG": "Doença de Chagas", "LEIV": "Leishmaniose visceral",
    "LTAN": "Leishmaniose tegumentar", "HEPA": "Hepatites virais",
    "SIFA": "Sífilis adquirida", "SIFC": "Sífilis congênita",
    "SIFG": "Sífilis em gestante", "COQU": "Coqueluche",
    "DIFT": "Difteria", "TETA": "Tétano acidental", "RAIV": "Raiva",
    "FMAC": "Febre maculosa", "FTIF": "Febre tifoide", "BOTU": "Botulismo",
    "COLE": "Cólera", "HANT": "Hantavirose", "INFL": "Influenza",
    "PEST": "Peste", "TOXC": "Toxoplasmose congênita",
    "TRAC": "Tracoma", "VARC": "Varicela", "PNEU": "Pneumoconiose",
    "PAIR": "Perda auditiva", "LERD": "LER/DORT", "DERM": "Dermatoses",
    "IEXO": "Intoxicação exógena", "MENT": "Transtorno mental",
    "NTRA": "Notificação de trabalho", "CANC": "Câncer relacionado ao trabalho",
    "SDTA": "Surto de doença transmitida por alimentos",
}

print(f"{len(agravos)} agravos com dados em {ANO}:\n")
for codigo in agravos:
    print(f"   {codigo:<6} {NOMES.get(codigo, '(consulte o Ministério da Saúde)')}")

56 agravos com dados em 2024:

   ACBI   Acidente biológico
   ACGR   Acidente de trabalho grave
   AIDA   (consulte o Ministério da Saúde)
   AIDC   (consulte o Ministério da Saúde)
   ANIM   Acidente por animal peçonhento
   ANTR   (consulte o Ministério da Saúde)
   BOTU   Botulismo
   CANC   Câncer relacionado ao trabalho
   CHAG   Doença de Chagas
   CHIK   Chikungunya
   COLE   Cólera
   COQU   Coqueluche
   DCRJ   (consulte o Ministério da Saúde)
   DENG   Dengue
   DERM   Dermatoses
   DIFT   Difteria
   ESQU   Esquistossomose
   EXAN   (consulte o Ministério da Saúde)
   FMAC   Febre maculosa
   FTIF   Febre tifoide
   HANS   Hanseníase
   HANT   Hantavirose
   HIVA   (consulte o Ministério da Saúde)
   HIVC   (consulte o Ministério da Saúde)
   HIVE   (consulte o Ministério da Saúde)
   HIVG   (consulte o Ministério da Saúde)
   IEXO   Intoxicação exógena
   LEIV   Leishmaniose visceral
   LEPT   Leptospirose
   LERD   LER/DORT
   LTAN   Leishmaniose tegumentar
   MALA   Malá

## Consultando qualquer recorte

A função abaixo responde à pergunta "existe dado de X?" para qualquer combinação:

In [6]:
def o_que_existe(base, **filtros):
    """Mostra o que há no catálogo para o recorte pedido."""
    catalogo = list_files(dataset=base, **filtros)
    if len(catalogo) == 0:
        print(f"❌ Nada em {base} com {filtros}")
        return
    nomes = sorted(n.split("\\")[-1] for n in catalogo["name"])
    print(f"✅ {len(nomes)} arquivo(s) em {base} com {filtros}")
    for nome in nomes[:8]:
        print(f"     {nome}")
    if len(nomes) > 8:
        print(f"     … e mais {len(nomes) - 8}")


o_que_existe("cnes", state="PR", year=2024, month=12)
print()
o_que_existe("sim", state="AM", year=2023)

✅ 12 arquivo(s) em cnes com {'state': 'PR', 'year': 2024, 'month': 12}
     DCPR2412.parquet
     EFPR2412.parquet
     EPPR2412.parquet
     EQPR2412.parquet
     GMPR2412.parquet
     HBPR2412.parquet
     INPR2412.parquet
     LTPR2412.parquet
     … e mais 4



✅ 1 arquivo(s) em sim com {'state': 'AM', 'year': 2023}
     DOAM2023.parquet


## Resumo prático

| Se você quer saber… | Use |
|---|---|
| Casos de uma doença | `sinan()` |
| Óbitos e suas causas | `sim()` |
| Nascimentos, cesáreas, peso | `sinasc()` |
| Internações, custos, permanência | `sih()` |
| Consultas, exames, procedimentos | `sia()` |
| Leitos, hospitais, profissionais | `cnes()` |
| Vacinação e cobertura | `pni()` |
| Atendimentos fora do SUS | `ciha()` |
| População (para calcular taxas) | `ibge()` |

---
*Notebook do projeto [PySusNoCode](https://github.com/cartaproale/PySusNoCode) —
um produto [Kraemer Academy](https://kraemeracademy.net).
Validado com dados reais do DATASUS.*